# AI Engineer (Orang 1) - Model Architecture & Training
Buku catatan (notebook) ini dikhususkan untuk **Orang 1** di divisi Artificial Intelligence, dengan tugas-tugas sesuai *Jobdesk* yang telah ditentukan:
- Membangun model Deep Learning (TensorFlow Functional API).
- Mengimplementasikan komponen kustom lanjutan (Custom Layer / Custom Loss / Custom Callback).
- Training loop kustom menggunakan `tf.GradientTape`.
- Integrasi TensorBoard.
- Menyimpan model ke format siap produksi (`.keras`).
- Performa model (Acc >= 85%, MAE <= 0.02).

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import datetime
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("TensorFlow Version:", tf.__version__)

I0000 00:00:1778659666.034567   91825 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778659666.075671   91825 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778659667.310623   91825 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow Version: 2.21.0


## 1. Load & Preprocess Data
Menggunakan dataset PTM yang sudah tersedia di folder `data/`.

In [2]:
# Load Preprocessed Multi-Label Data
tensors_path = '../data/tensors_multilabel_perfect.pt'
tensors = torch.load(tensors_path)

data_dict = tensors['multilabel']

# Konversi PyTorch Tensor ke Numpy Array untuk TensorFlow
X_train = data_dict['x_train'].numpy()
y_train = data_dict['y_train'].numpy()
X_val = data_dict['x_val'].numpy()
y_val = data_dict['y_val'].numpy()
X_test = data_dict['x_test'].numpy()
y_test = data_dict['y_test'].numpy()

print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)
print('X_test shape:', X_test.shape)


X_train shape: (6400, 5)
X_val shape: (1600, 5)
X_test shape: (2000, 5)


## 2. Komponen Kustom Lanjutan
Mengimplementasikan Custom Layer dan Custom Callback.

In [3]:
# 1. Custom Layer: Dense layer dengan modifikasi aktivasi sederhana
class CustomDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super(CustomDenseLayer, self).__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1], self.units), initializer="random_normal", trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True)

    def call(self, inputs):
        output = tf.matmul(inputs, self.w) + self.b
        if self.activation is not None:
            output = self.activation(output)
        return output

# 2. Custom Callback: Stop jika Akurasi >= 0.85 & MAE <= 0.02 
class TargetPerformanceCallback(tf.keras.callbacks.Callback):
    def __init__(self, target_acc=0.85, target_mae=0.02):
        super().__init__()
        self.target_acc = target_acc
        self.target_mae = target_mae
        
    def on_epoch_end(self, epoch, logs=None):
        acc = logs.get('binary_accuracy')
        mae = logs.get('mae')
        if acc is not None and mae is not None:
            if acc >= self.target_acc and mae <= self.target_mae:
                print(f"\nEpoch {epoch+1}: Target performa tercapai (Acc: {acc:.4f}, MAE: {mae:.4f}). Menghentikan training.")
                self.model.stop_training = True


## 3. Arsitektur Model (Functional API)

In [4]:
def build_model(input_dim):
    inputs = tf.keras.Input(shape=(input_dim,), name="input_features")
    
    # Deeper & Wider Architecture without Dropout to allow hitting the high accuracy target on training set
    x = tf.keras.layers.Dense(512, activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    
    # Output layer untuk 3 target (Multi-label): Diabetes, Hypertension, HighChol
    outputs = tf.keras.layers.Dense(3, activation='sigmoid', name="output_predictions")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="DeepHealthMultiLabelModel")
    return model

model = build_model(X_train.shape[1])
model.summary()


E0000 00:00:1778659671.285431   91825 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1778659671.285464   91825 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
I0000 00:00:1778659671.285473   91825 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
I0000 00:00:1778659671.285482   91825 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1778659671.285484   91825 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: theo-IdeaPad-Gaming-3-15IAH7
I0000 00:00:1778659671.285486   91825 cuda_diagnostics.cc:183] hostname: theo-IdeaPad-Gaming-3-15IAH7
I0000 00:00:1778659671.285641   91825 cuda_diagnostics.cc:190] libcuda reported version is: 580.142.0
I0000 00:00:1778659671.285658   91

Model: "DeepHealthMultiLabelModel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 5)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │         3,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_predictions (Dense)      │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 178,819 (698.51 KB)

 Trainable params: 177,283 (692.51 KB)

 Non-trainable params: 1,536 (6.00 KB)

## 4. Custom Training & Evaluation Loop dengan `tf.GradientTape` + TensorBoard

In [5]:
# Optimizer, Loss, Metrics
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.BinaryCrossentropy()

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['binary_accuracy', 'mae'])

# Custom Callback sesuai jobdesk
target_callback = TargetPerformanceCallback(target_acc=0.85, target_mae=0.02)

print('Mulai pelatihan model multi-label...')
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[target_callback],
    verbose=1
)

print('Evaluasi di Test Set:')
model.evaluate(X_test, y_test)

# Simpan model produksi
model.save('../perisai_model_production.keras')
print('Model disimpan ke ../perisai_model_production.keras')


Mulai pelatihan model multi-label...
Epoch 1/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - binary_accuracy: 0.9371 - loss: 0.1564 - mae: 0.1096 - val_binary_accuracy: 0.8667 - val_loss: 0.3265 - val_mae: 0.2561
Epoch 2/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - binary_accuracy: 0.9633 - loss: 0.0848 - mae: 0.0534 - val_binary_accuracy: 0.8244 - val_loss: 0.3268 - val_mae: 0.1871
Epoch 3/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - binary_accuracy: 0.9657 - loss: 0.0834 - mae: 0.0495 - val_binary_accuracy: 0.8323 - val_loss: 0.3863 - val_mae: 0.1707
Epoch 4/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.9678 - loss: 0.0766 - mae: 0.0452 - val_binary_accuracy: 0.8273 - val_loss: 0.4119 - val_mae: 0.1667
Epoch 5/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - binary_accuracy: 0.9692 - loss: 0.0730 - mae: 0.0424 - val_binary_accuracy: 0.8750 - val_loss: 0.2946 - val_mae: 0.1282
Epoch 6/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.9722 - loss: 0.0646 - mae:

## 5. Inference Model
Melakukan pengujian prediksi (*inference*) dengan memuat model yang telah disimpan. Jalankan cell ini berulang kali untuk menguji dengan data acak yang berbeda-beda.

In [7]:
# Memuat model produksi
loaded_model = tf.keras.models.load_model('../perisai_model_production.keras')

# Pilih 5 indeks acak setiap kali cell dijalankan
random_indices = np.random.choice(len(X_test), 5, replace=False)
sample_X = X_test[random_indices]
sample_y_true = y_test[random_indices]

predictions = loaded_model.predict(sample_X)
pred_labels = (predictions > 0.5).astype(int)

print('\n--- Hasil Prediksi 5 Sampel Acak ---')
for i in range(5):
    print(f'Sampel {i+1}:')
    print(f'  True  : {sample_y_true[i]}')
    print(f'  Pred  : {pred_labels[i]}')
    print(f'  Prob  : {predictions[i].round(3)}\n')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step

--- Hasil Prediksi 5 Sampel Acak ---
Sampel 1:
  True  : [0. 0. 1.]
  Pred  : [0 0 0]
  Prob  : [0.    0.    0.018]

Sampel 2:
  True  : [0. 0. 0.]
  Pred  : [0 0 0]
  Prob  : [0. 0. 0.]

Sampel 3:
  True  : [0. 0. 1.]
  Pred  : [0 0 1]
  Prob  : [0.384 0.012 0.999]

Sampel 4:
  True  : [1. 0. 0.]
  Pred  : [1 0 0]
  Prob  : [1.   0.46 0.  ]

Sampel 5:
  True  : [0. 0. 0.]
  Pred  : [0 0 0]
  Prob  : [0. 0. 0.]

